# Rotax 912 ULS - Step 5: How worn is the engine?

**In one sentence:** Estimates the engine's overall wear condition, from 1 (as new) to 0 (worn out), from how its readings compare with a healthy twin.

| Goes in | Comes out |
|---|---|
| The same 128-second window | One number from 0 to 1 |

Starts from what step 2 (*Which part is failing?*) already learned.

Run the cells top to bottom with **Shift+Enter**. If this step was already trained with `run.py`, training just loads the saved model in a second; otherwise it trains here.

### 0. Setup

In [ ]:
# Finds the project folder wherever Jupyter was started, then loads everything.
import os, sys
d = os.path.abspath('.')
for _ in range(6):
    if os.path.exists(os.path.join(d, 'nb_helpers.py')): break
    if os.path.exists(os.path.join(d, 'validation_v4', 'nb_helpers.py')):
        d = os.path.join(d, 'validation_v4'); break
    d = os.path.dirname(d)
sys.path.insert(0, d)
from nb_helpers import *

ENGINE, PHASE = '912', 'health'
setup(ENGINE, PHASE)

### 1. How wear builds up
Each dot is one second of flight. An engine's condition falls with age, and
falls faster when a fault is developing.

In [ ]:
d0 = load_rows(ENGINE, ["engine_hours", "health_index", "fault_present"], split="train")
df = d0.sample(min(20000, len(d0)), random_state=0)
plt.figure(figsize=(8, 3.6))
for v, col, lab in ((0, HEALTHY, "no fault"), (1, FAULTY, "with a fault")):
    d = df[df.fault_present == v]
    plt.scatter(d.engine_hours, d.health_index, s=2, alpha=0.4, color=col, label=lab)
plt.xlabel("engine hours"); plt.ylabel("wear condition (1 = new)"); plt.legend(markerscale=5)
plt.title("Condition falls with age, faster with a fault"); plt.tight_layout(); plt.show()

### 2. Train the model (or load it)

In [ ]:
payload = train_or_load(ENGINE, PHASE)
print(f"Done. Trained for {payload.get('epochs_run', '?')} passes over the data.")

### 3. Scorecard
Measured on the test flights. The quality bar catches a model that did not learn - it is a safety check, not a target.

In [ ]:
scorecard(ENGINE, PHASE)

### 4. See it working on flights it never saw
True condition against the model's estimate. Dots on the dashed line are exact.

In [ ]:
r = predict_test(ENGINE, PHASE)
true, pred = r["true"].ravel(), r["pred"].ravel()
plt.figure(figsize=(5, 5)); plt.scatter(true, pred, s=3, alpha=0.25, color=HEALTHY)
plt.plot([0, 1], [0, 1], "k--", lw=0.8); plt.xlim(0, 1.02); plt.ylim(0, 1.02)
plt.xlabel("true condition"); plt.ylabel("predicted condition")
plt.title("Each dot is one 128-second window"); plt.tight_layout(); plt.show()

### 5. What this means

In [ ]:
err = np.abs(pred - true)
print(f"Condition is off by {err.mean():.3f} on average on the 0-1 scale - "
      f"about {100 * err.mean():.0f} points out of 100.")

---
### Terms used

| Term | Meaning |
|---|---|
| **Window** | 128 seconds of flight data - the unit every model looks at. |
| **Twin** | A perfectly healthy simulated copy of the engine, flown through the exact same flight. |
| **Residual** | Real reading minus the healthy twin's reading. Near zero means the engine behaves as it should. |
| **Train / validation / test** | Flights used to learn / to pick the best version / to score it. A flight is only ever in one of them, so the score is on flights the model never saw. |
| **TBO** | Time between overhauls - the engine's scheduled life: 2000 h, or 1200 h for the 915. |
| **Warm start** | A step begins from what an earlier step already learned instead of from nothing. |